<a href="https://colab.research.google.com/github/rorisDS/workshop_ai_agents/blob/develop/notebooks_es/PrimerosPasosConLangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Primeros pasos con LangChain

En este notebook daremos los primeros pasos en la construcción de sistemas basados en **Modelos de Lenguaje (LLMs)** utilizando [**LangChain**](https://docs.langchain.com/oss/python/langchain/overview).

El objetivo **no es** construir un agente complejo desde el principio, sino **entender las piezas fundamentales** que hacen posible este tipo de sistemas:

- cómo se conecta un LLM,
- cómo se estructuran los mensajes,
- y cómo se definen herramientas (tools).

LangChain es una librería ampliamente utilizada en la industria y la comunidad, que proporciona **abstracciones claras** para trabajar con LLMs sin ocultar su funcionamiento interno. Esto la convierte en una buena opción para **aprender cómo funcionan los agentes de IA**, no solo para usarlos. Otras librerías similares son: [LlamaIndex](https://www.llamaindex.ai/), [Agents.js](https://www.npmjs.com/package/agents-js) o [OpenAI SDK](https://platform.openai.com/docs/guides/agents-sdk)

---
**¿Por qué no usar interfaces visuales?**

<img src="https://n8niostorageaccount.blob.core.windows.net/n8nio-strapi-blobs-prod/assets/Home_ITO_Ps_5a5aac3fda.webp" width="800px" />


Existen herramientas (como [n8n](https://n8n.io/), [zapier](https://zapier.com/) o [botpress](https://botpress.com/es/blog/build-ai-agent)) que permiten construir agentes mediante interfaces gráficas, conectando bloques sin escribir código. Aunque son útiles para prototipado rápido, **ocultan decisiones clave**:

- qué información recibe realmente el modelo,
- cuándo se ejecuta una herramienta,
- cómo se mantiene el contexto,
- y por qué el agente toma una decisión u otra.

En este workshop priorizamos la **comprensión del “cómo” y el “por qué”**, aunque eso implique escribir más código.

---
**Qué aprenderás en este notebook**

Al finalizar este notebook serás capaz de:

- Conectar un LLM usando LangChain
- Comprender el papel de los distintos tipos de mensajes (system, human, AI, tool)
- Ejecutar interacciones simples entre un usuario y un modelo
- Definir herramientas que un agente puede utilizar

Este notebook sirve como **base conceptual y técnica** para los siguientes, donde iremos profundizando en la creación y ejecución de agentes de IA usando los componentes aquí identificados.


## Instalación de librerías

In [9]:
!pip install langchain==1.2.7
!pip install langchain-core==1.2.7
!pip install langchain-openai==1.1.7  # Para usar modelos de OpenAI
!pip install langchain-google-genai==4.2.0  # Para usar modelos de Google (Gemini)
!pip install ddgs==9.10.0
!pip install langchain-community==0.4.1

# Limpia output
from IPython.display import clear_output
clear_output()

## Conectando con un LLM

LangChain proporciona abstracciones unificadas para conectarse a distintos Modelos de Lenguaje (LLMs), independientemente del proveedor subyacente (OpenAI, Google, etc.).  
Esto permite cambiar de modelo sin modificar la lógica principal de la aplicación. [Más informacion](https://docs.langchain.com/oss/python/langchain/models).

En la mayoría de los casos, el acceso a un LLM requiere una **API Key**, que identifica y autoriza las peticiones al servicio.

### Instanciando el cliente del LLM

Langchain ofrece clientes para la mayoría de proveedores de LLMs: [ver lista completa](https://docs.langchain.com/oss/python/integrations/providers/overview).

A continuación, presentamos a modo de ejemplo, la instanciación de dos de los proveedores más comunes: OpenAI y Google.

In [4]:
# Use Google Colab Secrets
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except:
    pass

* Conectando a un modelo de OpenAI

   - Crear API Key: https://platform.openai.com/docs/quickstart
   - Seleccionar un modelo: https://platform.openai.com/docs/models

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o",  # Seleccionamos un modelo por su keyword
    temperature=0,
    max_tokens=None,
    # other params...
    )

* Conectando a un modelo de Google
   - Crear API Key: https://ai.google.dev/gemini-api/docs/api-key
   - Seleccionar un modelo: https://ai.google.dev/gemini-api/docs/models

In [10]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    # model="gemini-2.5-flash",
    temperature=0.0,
    max_tokens=None,
    # other params...
)

### Llamada básica

Una vez instanciado el cliente del LLM, podemos interactuar con el modelo enviándole una consulta directa.
En este punto aún no estamos usando agentes, tools ni memoria: simplemente estamos enviando texto y recibiendo texto.

In [ ]:
response = llm.invoke("Hola, cómo estás?")
response

AIMessage(content=[{'type': 'text', 'text': '¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás? ¿En qué puedo ayudarte hoy?', 'extras': {'signature': 'EuMFCuAFAXLI2nxEDOGptImHy5Y2U0XsXM1hLILoHsuRQVtVI7ZWD481GcuaHAEqut9VR2WzXiiLUs9zcIhaVAKSzyoDtT7eWG7YGurVcHPqVNvIncUGIyx6GqSfliuFMu6275QexEf8J4CXmE3ezsdyPiIleuX7rbGfJ35oN+/QnnW8dGglH2gfR6kxe8xyz7uc6lms+ZqhpiBJ0cqS4XwWIxDBe5wclQlHBNj+Klm3UoiMgUN4LUwQlpjwYola/6UVB70OkM5k6piYT5kOkaORAe/34P5EWmHCagHRq96wB+xEVkQIqZ49LwHzgNCymIkhcgNvuspOQv0aeeqAUlRaBlXRGkUmbOa5QvCf9DreFyOFPPceHbb5rIhvLN1grz92wR4SOAdc3Ai2qSBEkVhcDZLpiRpzjzI0zHnOt42rhOXFbXBN9In4glCTUY1V9JAKchqP/ZnC9el8LCZlawYN7G8G78u/F/vogKtVkA6rgcQWwRBQC+gFLTnhV1kiYjwbXKnsgn191ZqwKsvWRp0S9XUr+ezmH3DCu4aI1/rZxiRlQvMYR7CLC+uTLtu4zx3yjgZwzRztjurvjhhoGyDfA6J7p2/HJl7LRw95UJuZ7UO/ar5pug9+iK05ArjYL3ZNU3mkaX4d+BgYnyplnxcTJTRsKL3zMn8VDuWadmHLavJr3AfZr4Gry3Y9X6x/lBihJV8xMdyyT6EZqJMVnLKPW6Kl+X/DRgrebnrNjPNPx3V1zjy+bDCbZtfhLIz7Iphk4eAFz/Ggq8Vt8npAVu4iyCes35vO0ydzpOnPhD6zp6sfZ3MvsXgPnDVkU2tIWEd

La respuesta del modelo no es un simple string, sino un objeto estructurado de tipo `AIMessage`.

Este objeto contiene:

- El **texto de respuesta**, pensado para el usuario final.
- Metadatos útiles para el desarrollador, como:
  - El modelo utilizado
  - El número de tokens consumidos
  - Información sobre el razonamiento interno del modelo

Información relevante para el usuario

In [ ]:
print(f"Respuesta: {response.content[0]['text']}")

Respuesta: ¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás? ¿En qué puedo ayudarte hoy?


Información relevante para el desarrollador

In [ ]:
print(f"\t - Respuesta del tipo: {type(response)}")
print(f"\t - LLM usado: {response.response_metadata['model_name']}")
print(f"\t - Tokens de entrada: {response.usage_metadata['input_tokens']}")
print(f"\t - Tokens generados: {response.usage_metadata['output_tokens']}")
print(f"\t - Tokens usados en razonamiento: {response.usage_metadata['output_token_details']['reasoning']}")

	 - Respuesta del tipo: <class 'langchain_core.messages.ai.AIMessage'>
	 - LLM usado: gemini-3-flash-preview
	 - Tokens de entrada: 6
	 - Tokens generados: 226
	 - Tokens usados en razonamiento: 201


### Visualizando el razonamiento del modelo

Los modelos con capacidad de razonamiento son capaces de **estructurar y expresar de forma lógica una secuencia de pasos o “pensamientos”** con el objetivo de mejorar la calidad de la respuesta final.

Algunas plataformas (por ejemplo, ciertos modelos ejecutados en AWS con capacidad de *reasoning*) pueden devolver un campo `reasoning` en los metadatos de la respuesta.  
Sin embargo, este comportamiento **no es estándar** y **no está disponible en todos los proveedores ni modelos**.

Modelos ampliamente utilizados como los de OpenAI o Gemini, aunque aplican mecanismos internos de razonamiento, **no exponen de forma nativa sus procesos internos de pensamiento a través de la API**.

Para poder observar este comportamiento de forma práctica, vamos a **simular el razonamiento** solicitando explícitamente al modelo que **explique sus pasos en lenguaje natural** antes de generar la respuesta final.  
Esta técnica se conoce como *chain of thought*.


In [ ]:
from langchain.messages import SystemMessage, HumanMessage

# Pedimos razonamiento paso a paso explícito
messages = [
    """
Para el siguiente problema, explica tus pasos de razonamiento antes de dar la respuesta final.

Problema:
---
Tengo 3 manzanas. Le doy 2 a Juan. Juan se come una y me devuelve la otra. \
Luego compro 5 manzanas más y le doy 3 a mi hermana que ya tenía 2. \
Mi hermana se come 4 y regala una a Juan. ¿Cuántas manzanas tengo?
---
    """
]

response = llm.invoke(messages)
print(response.content[0]['text'])

Para resolver este problema, seguiré el rastro de las manzanas que tú tienes en cada paso:

1.  **Inicio:** Tienes **3 manzanas**.
2.  **Le das 2 a Juan:** Te queda 1 manzana ($3 - 2 = 1$).
3.  **Juan te devuelve una:** Juan se come una, pero la que te devuelve se suma a tu cuenta. Ahora tienes **2 manzanas** ($1 + 1 = 2$).
4.  **Compras 5 manzanas más:** Sumas estas a las que ya tenías. Ahora tienes **7 manzanas** ($2 + 5 = 7$).
5.  **Le das 3 a tu hermana:** Restas estas de tu total. Ahora tienes **4 manzanas** ($7 - 3 = 4$).
6.  **Acciones de tu hermana:** El problema menciona que ella tenía 2, se come 4 y le da una a Juan. Sin embargo, **ninguna de estas acciones afecta la cantidad de manzanas que tú tienes en tu mano**, ya que esas manzanas ya habían salido de tu posesión.

**Respuesta final:**
Tienes **4 manzanas**.


Este ejercicio demuestra que los LLMs pueden **articular un proceso de pensamiento explícito** cuando se les pide.
Este tipo de salidas es útil para entender cómo un agente decide qué herramientas invocar o cómo descomponer un problema.

## Messages

Los mensajes son la capa que define **el contexto y el comportamiento** del LLM.
Un agente no se controla modificando el modelo, sino **controlando los mensajes que recibe**.

Cada mensaje tiene un rol específico dentro de la conversación:

- **System messages**  
  Definen cómo debe comportarse el modelo y qué reglas debe seguir.  
  En un agente, también se utilizan para describir las herramientas disponibles.

- **Human messages**  
  Representan los objetivos, preguntas o instrucciones del usuario.

- **AI messages**  
  Son las respuestas generadas por el modelo, incluyendo —en el caso de agentes— posibles llamadas a tools.



[Más información](https://docs.langchain.com/oss/python/langchain/messages)

### Cambiando el comportamiento del LLM
A continuación veremos cómo el **mismo mensaje del usuario** produce respuestas muy diferentes únicamente cambiando el *System Message*.


#### Ejemplo 1: Solo mensaje del usuario

En este caso no se proporciona ningún mensaje de sistema.  
El modelo responde utilizando su comportamiento por defecto.


In [ ]:
from langchain.messages import HumanMessage

messages = [
    HumanMessage("Hola, cómo estás?")
]

response = llm.invoke(messages)
print(f"Respuesta: {response.content[0]['text']}")

Respuesta: ¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás? ¿En qué puedo ayudarte hoy?


#### Ejemplo 2: Cambiando el rol del modelo

Introducimos un *System Message* para modificar el estilo de respuesta, sin cambiar el mensaje del usuario.

In [ ]:
from langchain.messages import SystemMessage

messages = [
    SystemMessage("Actúa como un escritor renacentista"),
    HumanMessage("Hola, cómo estás?")
]

response = llm.invoke(messages)
print(f"Respuesta: {response.content[0]['text']}")

Respuesta: ¡Salud y larga vida os sean concedidas, noble espíritu!

Me hallo, por la gracia de las Musas y la benevolencia de los astros, en un estado de fecunda quietud. Mi pluma no descansa, pues busco en la humilde tinta el reflejo de la verdad y la armonía que rigen este vasto y renovado universo. El sol de la razón ilumina mis jornadas, y aunque el tiempo es un río que corre presuroso hacia el olvido, trato de capturar su esencia en la belleza de la palabra y el estudio de los antiguos.

¿Y qué hay de vuestra merced? ¿Cómo os trata la Fortuna en esta jornada? ¿Acaso vuestro ánimo se encuentra colmado de virtudes, o buscáis, como este humilde servidor, el consuelo en la sabiduría y las artes? Decidme, pues mi oído está presto a escuchar vuestras nuevas.


#### Ejemplo 3: Misma pregunta, distinto comportamiento

El contenido del mensaje humano es idéntico.  
La única diferencia es el mensaje de sistema.

In [ ]:
messages = [
    SystemMessage("Actúa como un pirata de dibujos animados"),
    HumanMessage("Hola, cómo estás?")
]

response = llm.invoke(messages)
print(f"Respuesta: {response.content[0]['text']}")

Respuesta: ¡Ahoy, camarada de agua dulce! ¡Arrr! 🏴‍☠️🦜

Me encuentro de maravilla, ¡mejor que un loro con una galleta de oro! El viento sopla a mi favor, mi pata de palo no me chirría y mi brújula apunta directamente hacia la aventura. ¡Rayos y centellas, hoy es un gran día para surcar los siete mares!

¿Y tú qué tal, grumete? ¿Vienes en busca de tesoros escondidos o solo a saludar a este viejo lobo de mar? ¡Habla pronto o te haré caminar por la tabla! (¡Es broma, es broma, solo si no compartes tu botín!) ⚓💰🌊


#### Ejemplo 4: Reglas de comportamiento

El *System Message* también puede imponer restricciones explícitas sobre cómo debe responder el modelo.


In [ ]:
messages = [
    SystemMessage("Sé breve y conciso en todas tus respuestas."),
    HumanMessage("Hola, cómo estás?")
]

response = llm.invoke(messages)
print(f"Respuesta: {response.content[0]['text']}")

Respuesta: Hola. Muy bien, ¿y tú?


#### Ejemplo 5: Combinando múltiples instrucciones

Los mensajes de sistema pueden combinar reglas de estilo, formato y comportamiento.


In [ ]:
messages = [
    SystemMessage("Sé breve y conciso en todas tus respuestas (no más de 10 palabras). Evita palabras de menos de 5 letras."),
    HumanMessage("Hola, cómo estás?")
]

response = llm.invoke(messages)
print(f"Respuesta: {response.content[0]['text']}")

Respuesta: Estoy estupendo, gracias. Saludos cordiales.


#### Ejemplo 6: Asignar tarea

In [ ]:
messages = [
    SystemMessage("Traduce al inglés. Responde solo con la traducción."),
    HumanMessage("Hola, cómo estás?")
]

response = llm.invoke(messages)
print(f"Respuesta: {response.content[0]['text']}")

Respuesta: Hello, how are you?


Estos ejemplos muestran que el comportamiento del LLM no está “programado” en el modelo,
sino en el **contexto que construimos mediante mensajes**.

Este mecanismo es fundamental para entender cómo funcionan los agentes de IA.


### Mensajes multimodales


> Nota: este tipo de mensajes requiere modelos con capacidades multimodales.

Los mensajes que recibe un LLM no tienen por qué ser únicamente texto.
En modelos multimodales, un mensaje puede contener distintos tipos de información, como imágenes u otros formatos estructurados.

Desde el punto de vista de un agente, los mensajes actúan como **contenedores de información** que alimentan el proceso de decisión del modelo.


En el siguiente ejemplo, el mensaje del usuario combina texto e imagen dentro de un mismo `HumanMessage`. En este mensaje se solicita al LLM que titule la siguiente imagen:

<img src="https://thevirtualinstructor.com/blog/wp-content/uploads/2013/08/understanding-abstract-art.jpg" width="600px" />

In [ ]:
human_message = HumanMessage(content=[
    {"type": "text", "text": "Pon un título a la siguiente imagen"},
    {"type": "image_url", "image_url": {"url": "https://thevirtualinstructor.com/blog/wp-content/uploads/2013/08/understanding-abstract-art.jpg"}}
])

messages = [
    SystemMessage("Eres un experto analista gráfico especializado en arte abstracto."),
    human_message
]

response = llm.invoke(messages)
print(f"Respuesta: {response.content[0]['text']}")

Respuesta: Como analista de arte abstracto, observo en esta obra una poderosa tensión entre la calidez expansiva de los amarillos y la profundidad introspectiva de los azules y violetas. La textura matérica y las pinceladas gestuales sugieren una colisión entre elementos naturales.

Teniendo en cuenta su dinamismo y contraste, propongo el siguiente título:

### **"El Umbral del Horizonte"**

---
**Otras opciones según el enfoque del análisis:**

*   **Desde una perspectiva emocional:** *"Génesis de la Euforia"* (por la explosión de color central).
*   **Desde una perspectiva paisajística:** *"Amanecer Fragmentado"* (por la sugerencia de un sol que se descompone sobre el agua).
*   **Desde una perspectiva técnica:** *"Sinfonía de Materia y Luz"* (por el relieve y la luminosidad de la composición).


En este ejemplo:
- El **mensaje del sistema** define el rol y el conocimiento del modelo.
- El **mensaje humano** incluye tanto texto como una imagen.
- El LLM utiliza ambos tipos de información para generar su respuesta.


### Salidas estructuradas

Por defecto, un LLM genera texto libre pensado para ser leído por personas.
Sin embargo, en muchos sistemas —y especialmente en agentes de IA— necesitamos que la respuesta tenga una **estructura predecible**.

Las salidas estructuradas permiten definir explícitamente el formato de la respuesta,
de forma que el resultado pueda ser procesado automáticamente por otros componentes del sistema.

En un agente, la salida del modelo suele ser **entrada para otro proceso**.


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Task(BaseModel):
    description: str = Field(description="Short description of the task")
    priority: str = Field(
        description="Priority level of the task",
        enum=["LOW", "MEDIUM", "HIGH"]
    )

class TaskList(BaseModel):
    tasks: List[Task] = Field(description="List of identified tasks")

/tmp/ipython-input-3640448894.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'enum'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  priority: str = Field(


In [ ]:
model_with_structure = llm.with_structured_output(TaskList)

from langchain_core.messages import HumanMessage, SystemMessage

structured_output = model_with_structure.invoke([
    SystemMessage(
        content="You will receive a conversation. Your goal is to extract the tasks mentioned and classify their priority."
    ),
    HumanMessage(
        content="""
Tenemos que preparar la presentación para mañana, eso es urgente.
También estaría bien revisar el código cuando tengamos tiempo.
Ah, y no te olvides de enviar el email al profesor antes de final de semana.
"""
    )
])

tasks=[Task(description='Preparar la presentación para mañana', priority='HIGH'), Task(description='Revisar el código', priority='LOW'), Task(description='Enviar el email al profesor antes de final de semana', priority='MEDIUM')]


In [ ]:
for task in structured_output.tasks:
    print(f"Task: {task.description}, Priority: {task.priority}")

Task: Preparar la presentación para mañana, Priority: HIGH
Task: Revisar el código, Priority: LOW
Task: Enviar el email al profesor antes de final de semana, Priority: MEDIUM


En este ejemplo:
- El modelo no responde con texto libre.
- La salida sigue exactamente la estructura definida por el desarrollador.
- El resultado puede ser utilizado directamente por otro sistema o agente.

Las salidas estructuradas convierten al LLM en un componente fiable dentro de un sistema mayor.


## Tools

Las *tools* permiten que un LLM interactúe con el mundo exterior ejecutando funciones de Python.
Desde el punto de vista del modelo, una tool es simplemente una acción disponible que puede invocar cuando la necesita.

### Definición básica de una tool

En LangChain, una tool se define como una función estándar de Python anotada con el decorador `@tool`.

Una tool describe **qué hace**, **qué parámetros acepta** y **qué devuelve**.

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

print("Nombre:", multiply.name)
print("Descripcion:", multiply.description)
print("Esquema de argumentos:", multiply.args)

tool_response = multiply.invoke({"a": 2, "b": 3})
print(f"Respuesta de la tool: {tool_response}")

Nombre: multiply
Descripcion: Multiply two numbers.
Esquema de argumentos: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
Respuesta de la tool: 6


En este ejemplo:
- La función `multiply` se convierte en una tool disponible para el LLM.
- El decorador `@tool` extrae automáticamente el nombre, la descripción y los parámetros.
- La tool puede ejecutarse manualmente mediante `invoke`, igual que lo haría un modelo.

### Personalizando nombre y descripción

El nombre y la descripción son especialmente importantes,
ya que el LLM los utiliza para decidir cuándo invocar una tool.

In [ ]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    return str(eval(expression))

print("Nombre:", calc.name)
print("Descripcion:", calc.description)

tool_response = calc.invoke({"expression": "2*3"})
print(f"Respuesta de la tool: {tool_response}")

Nombre: calculator
Descripcion: Performs arithmetic calculations. Use this for any math problems.
Respuesta de la tool: 6


### Tooling disponible en LangChain

LangChain proporciona múltiples implementaciones de tools listas para usar.
Estas herramientas encapsulan accesos a sistemas externos y fuentes de información,
y pueden ser utilizadas posteriormente por agentes de IA.

* Herramientas de búsqueda y conocimiento:
  - **Tavily Search**
  - **SerpAPI**
  - **DuckDuckGo**
  - **Wikipedia**

* Herramientas de ejecución de código
  - **Python tool**
  - **REPL / Sandbox execution**

* Herramientas de bases de datos
  - **SQLDatabaseTool**
  - **Vector store tools**

* Herramientas de sistema y utilidades
  - Acceso a ficheros
  - Llamadas HTTP / APIs REST
  - Web scraping

Una lista con todas las tools accesibles (con su descripción e instrucciones de ejecución) es accesible en: [link](https://docs.langchain.com/oss/python/integrations/tools)

In [ ]:
# Ejemplo con el buscador DuckDuckGo: https://docs.langchain.com/oss/python/integrations/tools/ddg
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
print("Nombre:", search_tool.name)
print("Descripcion:", search_tool.description)
print("Esquema de argumentos:", search_tool.args)

Nombre: duckduckgo_search
Descripcion: A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
Esquema de argumentos: {'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


In [ ]:
tool_response = search_tool.invoke({'query': "DataSpartan España"})
print(f"Respuesta de la tool: {tool_response}")

Respuesta de la tool: Workshop impartido por DataSpartan sobre procesamiento de lenguaje natural (NLP), titulado «Texto a Conocimiento: Construyendo un Buscador Semántico con NLP», a las 9:30 el 11 de marzo en el Aula T109 de la Escuela de Ingeniería de Telecomunicaciones. Las grandes tecnológicas eligen España como el referente en el sur de Europa en centros de datos Es el único país donde Amazon, Microsoft, Meta, Google e IBM han anunciado planes. La ... Check DataSpartan in London, Finsbury Avenue on Cylex and find ☎ 020 7117 0..., contact info, ⌚ opening hours. Contribute to activebiz/ dataspartan development by creating an account on GitHub. En España existen cerca de 100 data centers entre construidos y proyectados y Siemens ofrece soluciones para el todo ciclo de vida de ambos modelos, además de estar presente en los grandes acuerdos mundiales con las principales compañías del sector.


### LLM y tools

En el siguiente ejemplo mostramos como se registran las tools accesibles a un LLM (instrucción `bind_tools`) y como este puede llegar a indicar a cúal y con qué parametros invocarla.

In [11]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Recupera el tiempo en una localizacion."""
    return f"El tiempo es soleado en {location}."

# Ligamos al llm las tools que tendra acceso
llm_con_tools = llm.bind_tools([get_weather])

# Ante una invocacion
response = llm_con_tools.invoke("¿Cúal es el tiempo en Vigo?")

# El LLM puede indicar llamadas a tools
for tool_call in response.tool_calls:
    # Visualiza las llamadas a tools hechas por el modelo
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Vigo'}


In [15]:
print(f"Tipo de mensage: {type(response)}")
print(f"Parametro tool calls: {response.tool_calls}")

Tipo de mensage: <class 'langchain_core.messages.ai.AIMessage'>
Parametro tool calls: [{'name': 'get_weather', 'args': {'location': 'Vigo'}, 'id': '95138ecb-a5b7-448c-8521-f07f201d3e58', 'type': 'tool_call'}]


## Conclusiones

En este notebook hemos recorrido los **componentes fundamentales** sobre los que se construyen los agentes de IA modernos, sin entrar todavía en la lógica completa de un agente.

Hemos visto que trabajar con LLMs va mucho más allá de enviar un texto y recibir una respuesta.
Un sistema basado en LLMs se compone de **piezas bien definidas**, cada una con una responsabilidad clara:
- **LLMs**: el motor de generación y razonamiento.
- **Mensajes**: el mecanismo que controla el comportamiento del modelo, define objetivos y mantiene el contexto.
- **Tools**: la forma de extender las capacidades del modelo más allá del texto, permitiéndole interactuar con el mundo exterior.

Estas piezas, por separado, ya permiten construir interacciones potentes.
Sin embargo, su verdadero potencial aparece cuando se **orquestan de forma coordinada**.

En este punto todavía:
- el modelo no decide por sí mismo cuándo usar una herramienta,
- no mantiene una estrategia a lo largo del tiempo,
- ni ejecuta acciones encadenadas de forma autónoma.

Y esto es intencional.

El objetivo de este notebook era **entender las bases**, no ocultarlas detrás de abstracciones de alto nivel.
Comprender qué información recibe el modelo, cómo se estructura una conversación y cómo se describen las acciones disponibles es esencial para entender **cómo y por qué funciona un agente de IA**.

En el siguiente notebook daremos el siguiente paso natural:
introducir una **capa de control** que permita al modelo decidir, razonar y actuar.

Es decir, construiremos **agentes de IA** a partir de los bloques que ya conoces.

> Un agente de IA es la combinación correcta de modelo, mensajes y herramientas,
coordinadas por una lógica de decisión.